# First steps with sentence embeddings

An embedding model turns a text into a vector, so that two texts with close meanings get close vectors. The model of these labs: [Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B), multilingual, 1024 dimensions, one of the best open models of its size.

1. Compare a dozen sentences
2. Search a month of press, with and without an instruction
3. Search in another language
4. Cosine, dot product, Euclidean distance
5. Truncate the vectors (Matryoshka)

This notebook needs a GPU: *Runtime > Change runtime type > T4 GPU*.

## Data and model

The passages of the 1914 French press (see the previous lab), and the model, loaded in half precision (`half()`): half the memory, faster on GPU.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device="cuda").half()
# The model accepts 32,000 tokens: we cut at 512, enough for our passages, otherwise
# the longest ones fill the GPU's memory.
model.max_seq_length = 512

## 1. Comparing a dozen sentences

Encode the sentences of `SENTENCES` with `model.encode_document`, compute the matrix of their similarities with `model.similarity`, and show it with `plt.imshow` (the sentences as axis labels). Which pairs get the best scores? Find two surprises.

In [ ]:
SENTENCES = [
    "The VPN keeps disconnecting",
    "I can't connect to the network from home",
    "Mon VPN se déconnecte sans arrêt",
    "I received a suspicious email",
    "A fraudulent message arrived in my inbox",
    "The printer stopped printing",
    "My screen stays black",
    "How do I book time off?",
    "When does the payslip arrive?",
    "The server works",
    "The server doesn't work",
    "The canteen closes at 2 pm",
]

In [ ]:
# Your code here

### Solution

In [ ]:
vectors = model.encode_document(SENTENCES)
similarities = model.similarity(vectors, vectors).numpy()

fig, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(similarities, cmap="viridis")
ax.set_xticks(range(len(SENTENCES)), SENTENCES, rotation=90)
ax.set_yticks(range(len(SENTENCES)), SENTENCES)
fig.colorbar(image)
plt.show()

# - A sentence and its French translation almost coincide: the model is multilingual.
# - "The server works" and "The server doesn't work" are very close: embeddings capture
#   the topic much better than a negation.
# - Two unrelated sentences don't drop to 0: only comparing scores makes sense.

## 2. Searching a month of press

Keep the passages published between July 20 and August 10, 1914 (`date` is a date), then encode their texts with `model.encode_document` (`batch_size=32`, `show_progress_bar=True`).

Then write a function `search(query, k=5, instruction=True)` that prints the date, the score and the beginning of the `k` passages closest to the query. With `instruction=True`, encode the query with `model.encode_query`, which puts an instruction before it ("*Given a web search query, retrieve relevant passages that answer the query*"); otherwise with `model.encode`, with no instruction.

In [ ]:
# Your code here

### Solution

In [ ]:
passages = pd.read_parquet("passages.parquet")
month = passages[passages.date.between("1914-07-20", "1914-08-10")].reset_index(drop=True)
print(len(month), "passages")
month_vectors = model.encode_document(month.text.tolist(), batch_size=32, show_progress_bar=True)


def search(query: str, k: int = 5, instruction: bool = True) -> None:
    query_vector = model.encode_query(query) if instruction else model.encode(query)
    scores = model.similarity(query_vector, month_vectors)[0].numpy()
    for i in np.argsort(-scores)[:k]:
        print(f"{month.date[i]:%d/%m} {scores[i]:.3f} {month.text[i][:150]}")

Search, with and without the instruction (the corpus is in French; the model searches across languages):

- the assassination of Jean Jaurès
- the general mobilization
- the verdict of Mme Caillaux's trial

Are the results relevant? Does the instruction change anything?

In [ ]:
# Your code here

### Solution

In [ ]:
for query in ("the assassination of Jean Jaurès", "the general mobilization", "the verdict of Mme Caillaux's trial"):
    for instruction in (True, False):
        print(f"--- {query} ({'with' if instruction else 'without'} instruction)")
        search(query, instruction=instruction)

# With the instruction, the query is encoded as a question looking for its answer: results
# are usually better, and their scores higher. Without it, a short query looks for texts
# that resemble it, not texts that answer it.

## 3. Searching in another language

The previous queries were in English, the corpus in French. Compare with the same queries in French (« l'assassinat de Jean Jaurès », « l'Allemagne déclare la guerre à la France »), and in another language you know.

In [ ]:
# Your code here

### Solution

In [ ]:
search("l'assassinat de Jean Jaurès")
search("l'Allemagne déclare la guerre à la France")
search("Deutschland erklärt Frankreich den Krieg")

## 4. Cosine, dot product, Euclidean distance

Compute the norm of the vectors in `month_vectors` (`np.linalg.norm`). Then, for the query "the general mobilization", rank the passages by cosine, dot product and Euclidean distance (closest first): do the three rankings have the same top 10? Why? When would that no longer be the case?

In [ ]:
# Your code here

### Solution

In [ ]:
vectors = month_vectors.astype(np.float32)
print(np.linalg.norm(vectors, axis=1)[:5])

query_vector = model.encode_query("the general mobilization").astype(np.float32)
dot = vectors @ query_vector
cosine = dot / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(query_vector))
euclidean = np.linalg.norm(vectors - query_vector, axis=1)
rankings = {
    "cosine": np.argsort(-cosine)[:10],
    "dot product": np.argsort(-dot)[:10],
    "Euclidean": np.argsort(euclidean)[:10],
}
for name, ranking in rankings.items():
    print(f"{name:12}", ranking)

# The model normalizes its vectors (norm 1): the dot product is then the cosine, and
# |u - v|² = 2 - 2 cos(u, v), so the Euclidean distance ranks in the same order. With
# vectors of different norms, the dot product would favor the longest ones.

## 5. Truncating the vectors (Matryoshka)

The model is trained so that the **first** dimensions carry most of the meaning. Truncate the passages' and queries' vectors to 512, 256, 128 and 64 dimensions (then renormalize them), and measure, for 200 random passages used as queries, the share of their 10 nearest neighbors (in 1024 dimensions) found again after truncation. How much memory would 1 million passages take at each size, in `float32`?

In [ ]:
# Your code here

### Solution

In [ ]:
def truncate(x: np.ndarray, dims: int) -> np.ndarray:
    x = x[:, :dims]
    return x / np.linalg.norm(x, axis=1, keepdims=True)


rng = np.random.default_rng(0)
queries = rng.choice(len(vectors), 200, replace=False)


def neighbours(x: np.ndarray) -> np.ndarray:
    scores = x[queries] @ x.T
    scores[np.arange(len(queries)), queries] = -np.inf  # a passage isn't its own neighbor
    return np.argsort(-scores, axis=1)[:, :10]


reference = neighbours(vectors)
for dims in (512, 256, 128, 64):
    found = neighbours(truncate(vectors, dims))
    overlap = np.mean([len(set(a) & set(b)) / 10 for a, b in zip(reference, found)])
    print(f"{dims:4} dimensions: {overlap:.0%} of neighbors found, {dims * 4 * 1e6 / 1e9:.1f} GB per million passages")